
# 大型语言模型（LLM）单词嵌入（Word Embedding）技术笔记

## 1. 简介与发展脉络

单词嵌入是将离散的文本符号（词、子词、字符）映射到连续的、低维的、实值向量空间的技术。其核心目标是让语义相近的词在向量空间中距离也更近。

嵌入技术的发展主要经历了两个阶段：
* **静态嵌入 (Static Embeddings):** 每个词有一个固定的向量表示，不随上下文改变。代表模型有 Word2Vec 和 GloVe。
* **动态/上下文嵌入 (Dynamic/Contextual Embeddings):** 词的向量表示根据其所在的上下文动态生成。这是现代 LLM（如 BERT、GPT）的核心。

---

## 2. 静态嵌入模型 (Static Embeddings)

### 2.1 Word2Vec

Word2Vec 是一种基于预测的无监督学习方法，核心是利用一个浅层神经网络来学习词向量。它有两种主要的架构：

#### 2.1.1 Skip-gram 架构

* **核心思想：** 给定一个中心词（input），预测其周围的上下文词（output）。
* **输入：** 中心词的 one-hot 编码。
* **输出：** 上下文词的概率分布。

#### 2.1.2 CBOW (Continuous Bag-of-Words) 架构

* **核心思想：** 给定周围的上下文词（input），预测中心的词（output）。
* **输入：** 上下文词向量的平均值或拼接值。
* **输出：** 中心词的概率分布。

#### 2.1.3 Word2Vec 的数学原理与公式

Word2Vec 实质上是一个分类任务。为了解决Softmax在大词表上的计算成本，常使用负采样（Negative Sampling）和层次Softmax进行优化。

以 **Skip-gram 结合负采样** 为例：

**目标：** 区分“真实的上下文词对”和“噪声词对”（随机采样的非上下文词）。

**目标函数（最大化似然）：**
对于中心词 $w_c$ 和真实上下文词 $w_t$（正样本），以及 $K$ 个随机采样的噪声词 $w_j$（负样本）：

$$J(\theta) = \log \sigma(v_{w_t}^T v_{w_c}) + \sum_{i=1}^{K} \log \sigma(-v_{w_{n_i}}^T v_{w_c})$$

**公式解读：**
* $\sigma$ 是 sigmoid 函数。
* $v_{w}$ 表示词 $w$ 的嵌入向量。
* 第一项 $v_{w_t}^T v_{w_c}$：最大化中心词和真实上下文词的内积，使其更相似。
* 第二项 $-v_{w_{n_i}}^T v_{w_c}$：最小化中心词和负样本的内积，使其更不相似。

---

### 2.2 GloVe (Global Vectors)

与 Word2Vec 不同，GloVe 是一种 **基于计数 (count-based)** 的模型。它利用全局的词共现统计信息来学习词向量。

#### 2.2.1 GloVe 的核心思想

GloVe 认为，词对的共现概率比值可以反映语义关系。例如，“ice”和“steam”各自与“solid”、“gas”等探针词的共现概率比值揭示了物理相态的区别。

#### 2.2.2 GloVe 的数学原理与公式

GloVe 的目标是使得两个词向量的点积可以近似反映它们在语料中的共现频率。

**目标函数（最小二乘损失）：**

$$J = \sum_{i,j=1}^{V} f(X_{ij}) (w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij})^2$$

**公式解读：**
* $V$ 是词表大小。
* $X_{ij}$ 表示词 $i$ 和词 $j$ 的共现次数。
* $w_i$ 和 $\tilde{w}_j$ 分别是词 $i$ 作为目标词和词 $j$ 作为上下文词的向量。
* $b_i$ 和 $\tilde{b}_j$ 是偏置项。
* $f(X_{ij})$ 是一个加权函数，用于平衡高频词和低频词的影响。
* 在语料库中，很多词对可能只出现一两次（噪声大），而像“of”、“the”这样的词出现频率极高（信息量低）。如果简单地使用平方误差，这些极高或极低频的词会主导损失函数。
---

### 2.3 静态嵌入的对比总结

| 维度 | Word2Vec | GloVe |
| :--- | :--- | :--- |
| **学习机制** | 局部上下文预测 (预测型模型) | 全局词共现矩阵分解 (基于计数的模型) |
| **训练数据** | 逐个滑动窗口读取数据 | 需要预先计算完整的共现矩阵 |
| **语义捕获** | 擅长捕获局部语义和线性关系 (如 $v(king)-v(man)+v(woman) \approx v(queen)$) | 擅长捕获全局语义关系和词义比值 |
| **局限性** | **多义词处理：** 无法区分同一个词在不同语料中的不同义项。 | **动态性：** 生成后即固定，无法根据当前上下文调整。 |

---

## 3. 现代基于 Transformer 的词嵌入机制

随着 LLM 的发展，Transformer 架构（如 BERT、GPT 系列）成为了主流。这些模型的输入嵌入不仅仅包含词的信息，而是由多种信息累加而成，且在模型内部动态演变。

### 3.1 动态上下文嵌入

LLM 的输入嵌入层（Embedding Layer）虽然在初始化时也是一个静态矩阵，但在模型训练过程中，通过大型神经网络基于上下文动态地产生词的表示。这意味着，同一个词在不同句子中会得到不同的向量，从而解决了多义词问题。

例如：
* “苹果（Apple）公司的股票上涨”
* “我吃了一个苹果（apple）”

在 BERT 中，“苹果”在这两句中的向量表示会完全不同。

### 3.2 Transformer 输入嵌入的构成

在 Transformer 模型（如 BERT）中，输入到一个 token 的最终嵌入表示是由以下三种向量**逐元素相加**得到的：

1.  **Token Embedding (词元嵌入):** 离散词元ID的向量表示。
2.  **Position Embedding (位置嵌入):** 赋予模型感知序列中各 token 位置信息的能力。
3.  **Segment Embedding (分段嵌入/句片嵌入):** 用于区分输入中的不同句子（特别是在 BERT 的 NSP 任务中）。GPT 等单向生成模型一般不使用。

**最终输入表示：**
$$\text{Input Embedding} = \text{Token Embedding} + \text{Position Embedding} + (\text{Optional}) \text{Segment Embedding}$$

---

### 3.3 训练任务与嵌入优化

在 Transformer 模型中，词嵌入不是直接以 Word2Vec 那样的目标来训练的，而是作为模型整体的一部分，通过预训练任务的误差反向传播间接优化。

#### 3.3.1 BERT 的双向训练任务

* **MLM (Masked Language Modeling):** 随机遮盖输入序列中15%的 Token，让模型根据上下文预测这些被遮盖的词。这迫使 Token 嵌入和 Transformer 层捕获丰富的双向语义信息。
* **NSP (Next Sentence Prediction):** 判断句子B是否是句子A的下一句。该任务有助于优化 Segment Embedding 和全局语义捕获。

#### 3.3.2 GPT 系列的单向训练任务

* **Causal Language Modeling (自回归语言模型):** 给定前文预测下一个词。模型以单向方式训练，强制模型仅通过过去的上下文来调整 embedding。

---

## 4. PyTorch 实现与应用

在实际开发中，静态嵌入常通过直接加载预训练权重来使用，而动态嵌入则是模型内部结构的一部分。

### 4.1 PyTorch 中的词嵌入层 (静态嵌入)


In [1]:

import torch
import torch.nn as nn

# 1. 静态嵌入的简单实现
# 假设词表大小为 1000，嵌入维度为 128
vocab_size = 1000
embedding_dim = 128
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# 输入一个 batch 的 token ID (假设 batch_size=2, sequence_length=5)
input_ids = torch.LongTensor([[1, 5, 10, 20, 100], [2, 7, 30, 40, 50]])

# 获取词嵌入
word_embeddings = embedding_layer(input_ids)

# 输出形状: [batch_size, sequence_length, embedding_dim]
print(f"Shape of input ids: {input_ids.shape}")
print(f"Shape of word embeddings: {word_embeddings.shape}") 
# 预期输出: Shape of word embeddings: torch.Size([2, 5, 128])

Shape of input ids: torch.Size([2, 5])
Shape of word embeddings: torch.Size([2, 5, 128])



### 4.2 LLM (如 BERT) 的完整输入嵌入 (动态嵌入)

现代 LLM 的输入嵌入是一个更复杂的模块，包含了词元、位置和分段。我们可以使用 `transformers` 库轻松获取。

In [2]:
%env HF_ENDPOINT=https://hf-mirror.com

env: HF_ENDPOINT=https://hf-mirror.com


In [3]:

from transformers import BertTokenizer, BertModel
import torch

# 1. 加载预训练的 Tokenizer 和 Model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 2. 准备文本
text = "The apple company stock rose. I ate an apple."

# 3. Tokenization 和 编码
inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
input_ids = inputs['input_ids']
# 注意：BERT 会自动添加 [CLS] 和 [SEP] 标记，以及生成 token_type_ids (segment embeddings)

print(f"Token IDs: {input_ids}")

# 4. 前向传播获取嵌入
with torch.no_grad():
    outputs = model(**inputs)

# outputs 包含 last_hidden_state, pooler_output 等
# last_hidden_state 是 Transformer 层最后的输出，是动态上下文嵌入
last_hidden_state = outputs.last_hidden_state 

# 5. 查看特定 Token 的上下文嵌入
# 在句子中，"apple" 的 token ID 是一样的。我们比较两个句子中 "apple" 的向量。
# 注意：实际操作中，需要找到 token ID 在序列中的对应位置。
# 找到文本中 "apple" 的 token ID
apple_token_id = tokenizer.convert_tokens_to_ids("apple")
print(f"Token ID for 'apple': {apple_token_id}")

# 找到 batch 中 apple token ID 的位置 (这里演示找第一个)
# 实际操作需要准确的 index 映射
apple_indices = (input_ids == apple_token_id).nonzero(as_tuple=True)
print(f"Indices of 'apple' tokens: {apple_indices}")

# 获取第一个 "apple" 和第二个 "apple" 的向量表示
apple_vector_1 = last_hidden_state[apple_indices[0][0], apple_indices[1][0]]
apple_vector_2 = last_hidden_state[apple_indices[0][1], apple_indices[1][1]]

# 6. 比较两个向量的相似度 (余弦相似度)
cos = torch.nn.CosineSimilarity(dim=0)
similarity = cos(apple_vector_1, apple_vector_2)

print(f"Similarity between two 'apple' context vectors: {similarity.item()}")
# 预期输出：两个向量不完全相同，相似度会比较高但不是1.0，因为它们捕捉了不同的上下文信息。



/t9k/mnt/.conda/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1382.07it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different tas

Token IDs: tensor([[ 101, 1996, 6207, 2194, 4518, 3123, 1012, 1045, 8823, 2019, 6207, 1012,
          102]])
Token ID for 'apple': 6207
Indices of 'apple' tokens: (tensor([0, 0]), tensor([ 2, 10]))
Similarity between two 'apple' context vectors: 0.8138578534126282



## 5. 总结

| 特性 | 静态嵌入 (Word2Vec, GloVe) | 动态嵌入 (BERT, GPT) |
| :--- | :--- | :--- |
| **词表示** | 一个词一个固定向量 | 随上下文变化 |
| **计算成本** | 低，预训练后可直接查表 | 高，需要运行大型神经网络 |
| **多义词处理** | 无法处理 | 完美处理 |
| **下游任务效果** | 较好，通常作为输入层 | 卓越，通常作为微调或特征提取 |
| **未来方向** | 在资源受限场景下仍有应用空间 | 结合子词方案 (Subword Embeddings) 已成为绝对主流，不断探索更深层的语义捕获 |